# 14. 画像の可視化及び手術支援としての画像活用 ― 演習

## 3D可視化を「作って」理解する：レイキャスティング法の実装

**臨床工学科／医療科学部　演習用ノートブック**

---

講義では、CTやMRIの2D断層画像を3Dの立体像として可視化する技術を学びました。
この演習では、その中核をなす **レイキャスティング法 (Ray-Casting)** を
**自分たちの手で実装したレンダラ**で動かし、以下を体験的に理解します。

| 講義の内容 | 本演習で確かめること |
|---|---|
| **ボリュームレンダリング (VR)** | 透明度を設定すると、なぜ**内部が透けて見える**のか |
| **サーフェスレンダリング (SR)** | 閾値ひとつで形が変わる。なぜ**内部が見えない**のか |
| **伝達関数 (Transfer Function)** | 閾値・不透明度を変えると描出がどう変わるか |
| **手術シミュレーション（仮想切除）** | 3Dボリュームを**切断**し、切断面に2D断層像を出す |
| **画像座標系と物理座標系** | ボクセルサイズ (spacing) を無視すると形が歪む |

### 使用するデータ

| ファイル | 内容 |
|---|---|
| `T1c.nii.gz` | 造影T1強調画像（脳MRI） |
| `tumor_ROI.nii.gz` | 腫瘍のマスク画像（1, 2, 3 … のマルチラベル） |

> 2つのファイルを、左のファイルペインから Colab にアップロードしてください
> （またはGoogle Driveをマウントしてパスを書き換えてください）。

### 本ツールの構成

```
   NIfTI (nibabel)  →  C++/CUDA レイキャスタ  →  画像  →  ブラウザ (マウス操作)
        Python              GPU で並列計算            HTML5 canvas
```

各画素から光線 (Ray) を飛ばして色と不透明度を合成する、という講義そのままの
アルゴリズムを **CUDA で1画素=1スレッドとして並列実行**します。

> ⚠️ **最初に必ず**「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択してください。
> （切り替えるとランタイムが再起動し、変数もアップロードしたファイルも消えます）

---
## 1. 準備：ライブラリのインストール

- `nibabel` … NIfTI形式 (.nii.gz) の読み込み。**ヘッダーからボクセルサイズを取得**するために重要。
- `scipy` … ROIマスクの平滑化（表面をなめらかにする）
- `pillow` … 描画結果をブラウザへ転送するための画像エンコード

In [ ]:
!pip install -q nibabel scipy pillow

---
## 2. レンダラ本体（CUDA / C++）を書き出す

ここが本演習の心臓部です。**講義で学んだアルゴリズムがそのままコードになっています。**
`%%writefile` でC++（CUDA）のソースコードをファイルに書き出します。

### 2.1 レイキャスティング法の実装（`kRender` カーネル）

講義の手順と、コードの対応は次の通りです。

| 講義の手順 | コード上の実装 |
|---|---|
| ① 画面の各画素から光線を飛ばす | `px`, `py` から光線の原点 `o` と方向 `dir` を計算（**1画素=1 CUDAスレッド**） |
| ② 光線に沿ってボクセル値を読む | `for(float t=tstart; t<=tend; t+=step)` のループ内で `sampleTex()` |
| ③ 色と不透明度を蓄積・合成する | `accR += (1-accA)*a*col; accA += (1-accA)*a;` （**front-to-back 合成**） |

> **平行投影法か、透視投影法か？**
> 本実装はすべての光線が平行に進む **平行投影法** です（`dir` が全画素で共通）。
> 仮想内視鏡のような透視投影にするには、光線を一点から放射状に出すよう `dir` を
> 画素ごとに変える必要があります（**発展課題**）。

### 2.2 伝達関数 (Transfer Function) はどこにあるか

VRのパートに注目してください。ボクセル値 `nv`（0〜1に正規化した信号強度）を
**不透明度 `a` に変換している式**、これが最も単純な伝達関数です。

```cpp
float frac = (nv - P.vr_thr) / (1.0f - P.vr_thr);
float a    = P.vr_opacity * frac * frac;   // ← これが伝達関数（2次関数）
```

閾値 `vr_thr` より暗いボクセルは完全に透明（`a=0`）、明るいほど不透明になります。
「職人技」と言われる伝達関数の設計を、あとで実際にスライダーで体感します。

### 2.3 サーフェスレンダリングの実装

講義では **Marching Cubes法**（ポリゴンを生成する方式）を学びましたが、
本実装は **レイキャスト中に等値面との交差を直接検出する**方式をとっています。

```cpp
if(prevT1 < P.sr_iso && nv >= P.sr_iso){ ... }   // 閾値を跨いだ瞬間＝表面
```

ポリゴンを作らない点は異なりますが、**「閾値で表面を決め、不透明に描く」**という
考え方と、**閾値の設定次第で結果が激変する**という性質は全く同じです。
表面の陰影は、信号強度の**勾配 (gradient) を法線ベクトル**とみなして計算しています。

### 2.4 物理座標系の維持

`world = voxel_index × spacing` として、**ボクセルサイズ (mm) を掛けてから**
光線を飛ばしています（`sampleTex()` 内の `p.x/sx` など）。
これを怠ると、スライス厚が面内解像度と異なるデータで**脳が縦や横に潰れて表示されます**。
講義11の「画像座標系 ↔ 物理座標系」の話と同じ問題です。

In [ ]:
%%writefile renderer.cu
// renderer.cu  -- GPU (CUDA) software ray-caster
// Same maths as the CPU version, but:
//   * volumes are uploaded ONCE to 3D textures (gpu_init) and reused for every
//     redraw, so slider/checkbox updates only pay for the ray-cast itself
//   * trilinear interpolation is done by the texture units in hardware (free)
//   * one CUDA thread per output pixel
//
// nvcc -O3 -Xcompiler -fPIC -shared renderer.cu -o librenderer.so
//
// API (ctypes):
//   gpu_init(t1, nx,ny,nz, roi_fields, n_labels, label_colors) -> 0 on success
//   render(...)      same signature/semantics as the CPU version
//   gpu_free()

#include <cuda_runtime.h>
#include <cstdio>
#include <cmath>

#define MAXL 8   // max ROI labels

// ---------- device-side small vector helpers ----------
struct V3 { float x, y, z; };
__host__ __device__ static inline V3 mk(float x,float y,float z){ V3 v; v.x=x;v.y=y;v.z=z; return v; }
__device__ static inline V3 add(V3 a,V3 b){ return mk(a.x+b.x,a.y+b.y,a.z+b.z); }
__device__ static inline V3 mul(V3 a,float s){ return mk(a.x*s,a.y*s,a.z*s); }
__device__ static inline float dot3(V3 a,V3 b){ return a.x*b.x+a.y*b.y+a.z*b.z; }
__device__ static inline V3 cross3(V3 a,V3 b){ return mk(a.y*b.z-a.z*b.y, a.z*b.x-a.x*b.z, a.x*b.y-a.y*b.x); }
__device__ static inline V3 nrm(V3 a){ float l=sqrtf(dot3(a,a)); return l>1e-12f? mul(a,1.0f/l):a; }

// ---------- persistent GPU state ----------
static cudaArray_t        d_t1Arr   = nullptr;
static cudaTextureObject_t h_t1Tex  = 0;
static cudaArray_t        d_roiArr[MAXL]  = {nullptr};
static cudaTextureObject_t h_roiTex[MAXL] = {0};
static unsigned char*     d_img     = nullptr;
static int   g_nx=0, g_ny=0, g_nz=0;
static size_t g_imgBytes = 0;

// Textures are addressed in *voxel index* space with unnormalised coords.
// tex3D<float>(tex, x+0.5, y+0.5, z+0.5) samples voxel-centre correctly.
__device__ static inline float sampleTex(cudaTextureObject_t tex, V3 p,
                                         float sx,float sy,float sz,
                                         int nx,int ny,int nz)
{
    float fx = p.x/sx, fy = p.y/sy, fz = p.z/sz;   // fx->i, fy->j, fz->k
    if(fx<0.f||fy<0.f||fz<0.f||fx>nx-1.f||fy>ny-1.f||fz>nz-1.f) return 0.f;
    // host layout idx=(i*ny+j)*nz+k  => k is the fastest dim => texture width.
    // tex3D takes (width, height, depth) = (k, j, i).
    return tex3D<float>(tex, fz+0.5f, fy+0.5f, fx+0.5f);
}

__device__ static inline V3 gradTex(cudaTextureObject_t tex, V3 p,
                                    float sx,float sy,float sz,int nx,int ny,int nz)
{
    float h = fminf(fminf(sx,sy),sz);
    float gx = sampleTex(tex,mk(p.x+h,p.y,p.z),sx,sy,sz,nx,ny,nz)
             - sampleTex(tex,mk(p.x-h,p.y,p.z),sx,sy,sz,nx,ny,nz);
    float gy = sampleTex(tex,mk(p.x,p.y+h,p.z),sx,sy,sz,nx,ny,nz)
             - sampleTex(tex,mk(p.x,p.y-h,p.z),sx,sy,sz,nx,ny,nz);
    float gz = sampleTex(tex,mk(p.x,p.y,p.z+h),sx,sy,sz,nx,ny,nz)
             - sampleTex(tex,mk(p.x,p.y,p.z-h),sx,sy,sz,nx,ny,nz);
    return mk(gx,gy,gz);
}

__device__ static inline bool boxHit(V3 o,V3 d,V3 ext,float& t0,float& t1)
{
    t0=-1e30f; t1=1e30f;
    #pragma unroll
    for(int a=0;a<3;a++){
        float oo = (a==0?o.x:(a==1?o.y:o.z));
        float dd = (a==0?d.x:(a==1?d.y:d.z));
        float mx = (a==0?ext.x:(a==1?ext.y:ext.z));
        if(fabsf(dd)<1e-12f){ if(oo<0.f||oo>mx) return false; }
        else{
            float ta=(0.f-oo)/dd, tb=(mx-oo)/dd;
            if(ta>tb){ float tt=ta; ta=tb; tb=tt; }
            t0=fmaxf(t0,ta); t1=fminf(t1,tb);
            if(t0>t1) return false;
        }
    }
    return t1>0.f;
}

struct Params {
    int nx,ny,nz; float sx,sy,sz;
    int n_labels;
    float lc[3*MAXL];
    cudaTextureObject_t roiTex[MAXL];
    cudaTextureObject_t t1Tex;
    float azim,elev;
    int show_vr, show_sr, show_roi, show_clip;
    float t1_lo, t1_hi, sr_iso, vr_thr, vr_opacity, clip_k;
    float zoom, pan_u, pan_v;
    int W,H;
};

__global__ void kRender(Params P, unsigned char* __restrict__ out)
{
    int px = blockIdx.x*blockDim.x + threadIdx.x;
    int py = blockIdx.y*blockDim.y + threadIdx.y;
    if(px>=P.W || py>=P.H) return;

    const int nx=P.nx, ny=P.ny, nz=P.nz;
    const float sx=P.sx, sy=P.sy, sz=P.sz;
    V3 ext = mk((nx-1)*sx,(ny-1)*sy,(nz-1)*sz);
    V3 C   = mul(ext,0.5f);
    float diag = sqrtf(dot3(ext,ext));
    float inv_range = (P.t1_hi>P.t1_lo)? 1.0f/(P.t1_hi-P.t1_lo) : 1.0f;

    V3 eye = mk(cosf(P.elev)*sinf(P.azim), cosf(P.elev)*cosf(P.azim), sinf(P.elev));
    V3 f   = mul(eye,-1.0f);
    V3 upW = (fabsf(eye.z)>0.95f)? mk(0,1,0) : mk(0,0,1);
    V3 right = nrm(cross3(upW,f));
    V3 up    = nrm(cross3(f,right));
    V3 light = nrm(add(mul(f,-1.0f), mul(right,0.35f)));

    float zoom = fmaxf(P.zoom, 1e-3f);
    float pxw  = diag*1.25f/ (float)max(P.W,P.H) / zoom;   // zoom
    float Rcam = diag*1.2f;
    float panW_u = P.pan_u*diag*1.25f/zoom;                // pan (screen->world)
    float panW_v = P.pan_v*diag*1.25f/zoom;

    float u = (px - P.W*0.5f + 0.5f)*pxw - panW_u;
    float w = (P.H*0.5f - py - 0.5f)*pxw - panW_v;
    V3 o   = add(add(add(C, mul(eye,Rcam)), mul(right,u)), mul(up,w));
    V3 dir = f;

    size_t oi = ((size_t)py*P.W + px)*3;
    out[oi]=0; out[oi+1]=0; out[oi+2]=0;

    float tn,tf;
    if(!boxHit(o,dir,ext,tn,tf)) return;
    if(tn<0.f) tn=0.f;

    float zc = P.clip_k*sz;
    bool keepUpper = (eye.z < 0.0f);
    float step = fminf(fminf(sx,sy),sz)*0.6f;

    float tstart=tn, tend=tf;
    bool entryAtPlane=false;
    if(P.show_clip){
        float dz=dir.z, oz=o.z;
        if(fabsf(dz)<1e-12f){
            bool inside = keepUpper? (oz>=zc):(oz<=zc);
            if(!inside) return;
        } else {
            float tc=(zc-oz)/dz;
            if(keepUpper){
                if(dz>0.f){ if(tc>tstart){ tstart=tc; entryAtPlane=true; } }
                else      { if(tc<tend) tend=tc; }
            } else {
                if(dz<0.f){ if(tc>tstart){ tstart=tc; entryAtPlane=true; } }
                else      { if(tc<tend) tend=tc; }
            }
        }
    }
    if(tstart>=tend) return;

    float accR=0.f, accG=0.f, accB=0.f, accA=0.f;
    bool done=false;

    // ---- cut face: crisp 2D slice painted where the plane meets tissue ----
    if(P.show_clip && entryAtPlane){
        V3 p = add(o, mul(dir,tstart));
        float raw = sampleTex(P.t1Tex,p,sx,sy,sz,nx,ny,nz);
        float nv  = fminf(fmaxf((raw-P.t1_lo)*inv_range,0.f),1.f);
        int lab=-1; float best=0.5f;
        for(int L=0; L<P.n_labels; ++L){
            float pr = sampleTex(P.roiTex[L],p,sx,sy,sz,nx,ny,nz);
            if(pr>best){ best=pr; lab=L; }
        }
        bool tissue = nv > 0.10f;
        if(P.show_roi && lab>=0){
            float s = 0.55f + 0.45f*nv;
            accR=P.lc[3*lab+0]*s; accG=P.lc[3*lab+1]*s; accB=P.lc[3*lab+2]*s;
            accA=1.f; done=true;
        } else if(tissue){
            accR=accG=accB=nv; accA=1.f; done=true;
        }
        if(done){
            out[oi]  =(unsigned char)fminf(255.f,accR*255.f);
            out[oi+1]=(unsigned char)fminf(255.f,accG*255.f);
            out[oi+2]=(unsigned char)fminf(255.f,accB*255.f);
            return;
        }
    }

    // ---- march ----
    V3 p0 = add(o,mul(dir,tstart));
    float prevT1 = (sampleTex(P.t1Tex,p0,sx,sy,sz,nx,ny,nz)-P.t1_lo)*inv_range;
    float prevL[MAXL];
    for(int L=0;L<P.n_labels;++L) prevL[L]=sampleTex(P.roiTex[L],p0,sx,sy,sz,nx,ny,nz);

    float voxmin = fminf(fminf(sx,sy),sz);

    for(float t=tstart+step; t<=tend && !done; t+=step){
        if(accA>0.985f) break;
        V3 p = add(o, mul(dir,t));
        float raw = sampleTex(P.t1Tex,p,sx,sy,sz,nx,ny,nz);
        float nv  = fminf(fmaxf((raw-P.t1_lo)*inv_range,0.f),1.f);

        if(P.show_sr){
            if(prevT1 < P.sr_iso && nv >= P.sr_iso){
                V3 N = nrm(mul(gradTex(P.t1Tex,p,sx,sy,sz,nx,ny,nz),-1.0f));
                float diff = fmaxf(0.f, dot3(N,light));
                float col = 0.25f + 0.75f*diff;
                accR += (1.f-accA)*col; accG += (1.f-accA)*col; accB += (1.f-accA)*col;
                accA += (1.f-accA);
                done = (accA>0.985f);
            }
        }
        if(P.show_roi){
            for(int L=0; L<P.n_labels; ++L){
                float cur = sampleTex(P.roiTex[L],p,sx,sy,sz,nx,ny,nz);
                if(prevL[L] < 0.5f && cur >= 0.5f){
                    V3 N = nrm(mul(gradTex(P.roiTex[L],p,sx,sy,sz,nx,ny,nz),-1.0f));
                    float diff = fmaxf(0.f, dot3(N,light));
                    float shade = 0.30f + 0.70f*diff;
                    accR += (1.f-accA)*P.lc[3*L+0]*shade;
                    accG += (1.f-accA)*P.lc[3*L+1]*shade;
                    accB += (1.f-accA)*P.lc[3*L+2]*shade;
                    accA += (1.f-accA);
                }
                prevL[L]=cur;
            }
            if(accA>0.985f) done=true;
        }
        if(P.show_vr && !done){
            if(nv>P.vr_thr){
                float frac=(nv-P.vr_thr)/fmaxf(1e-6f,1.0f-P.vr_thr);
                float a = P.vr_opacity*frac*frac;
                a = 1.0f - powf(1.0f-a, step/voxmin);          // opacity correction
                V3 g = gradTex(P.t1Tex,p,sx,sy,sz,nx,ny,nz);
                float gl = sqrtf(dot3(g,g));
                V3 N = nrm(mul(g,-1.0f));
                float diff = (gl>1e-6f)? fmaxf(0.f,dot3(N,light)) : 1.0f;
                float col = nv*(0.45f + 0.55f*diff);
                accR += (1.f-accA)*a*col; accG += (1.f-accA)*a*col; accB += (1.f-accA)*a*col;
                accA += (1.f-accA)*a;
            }
        }
        prevT1 = nv;
    }

    out[oi]  =(unsigned char)fminf(255.f,accR*255.f);
    out[oi+1]=(unsigned char)fminf(255.f,accG*255.f);
    out[oi+2]=(unsigned char)fminf(255.f,accB*255.f);
}

// ---------- helper: upload one float volume into a 3D texture ----------
static bool uploadVolume(const float* host, int nx,int ny,int nz,
                         cudaArray_t& arr, cudaTextureObject_t& tex)
{
    cudaChannelFormatDesc ch = cudaCreateChannelDesc<float>();
    // extent is in (width=x, height=y, depth=z); our host layout is
    // idx=(i*ny+j)*nz+k  => k (z) is fastest -> so "width" must be nz.
    cudaExtent ext = make_cudaExtent(nz, ny, nx);
    if(cudaMalloc3DArray(&arr, &ch, ext) != cudaSuccess) return false;

    cudaMemcpy3DParms cp = {0};
    cp.srcPtr   = make_cudaPitchedPtr((void*)host, nz*sizeof(float), nz, ny);
    cp.dstArray = arr;
    cp.extent   = ext;
    cp.kind     = cudaMemcpyHostToDevice;
    if(cudaMemcpy3D(&cp) != cudaSuccess) return false;

    cudaResourceDesc rd = {};
    rd.resType = cudaResourceTypeArray;
    rd.res.array.array = arr;
    cudaTextureDesc td = {};
    td.addressMode[0]=cudaAddressModeClamp;
    td.addressMode[1]=cudaAddressModeClamp;
    td.addressMode[2]=cudaAddressModeClamp;
    td.filterMode      = cudaFilterModeLinear;   // hardware trilinear
    td.readMode        = cudaReadModeElementType;
    td.normalizedCoords= 0;
    return cudaCreateTextureObject(&tex,&rd,&td,nullptr) == cudaSuccess;
}

extern "C" {

// NOTE: texture coords are (x=fastest dim). Our world axes map as
//   world x -> i (slowest), y -> j, z -> k (fastest)
// so in tex3D we must pass (k, j, i) = (z, y, x). Handled in sampleTexXfm below
// by the wrapper: we simply swap the argument order at the call site.
// (see sampleTex: tex3D<float>(tex, fx, fy, fz) -- we therefore build the
//  texture with extent (nz,ny,nx) and call tex3D(tex, k, j, i).)

int gpu_init(const float* t1, int nx, int ny, int nz,
             const float* roi_fields, int n_labels, const float* label_colors)
{
    (void)label_colors;
    if(n_labels > MAXL) return -2;
    // free anything previous
    if(h_t1Tex){ cudaDestroyTextureObject(h_t1Tex); h_t1Tex=0; }
    if(d_t1Arr){ cudaFreeArray(d_t1Arr); d_t1Arr=nullptr; }
    for(int L=0;L<MAXL;L++){
        if(h_roiTex[L]){ cudaDestroyTextureObject(h_roiTex[L]); h_roiTex[L]=0; }
        if(d_roiArr[L]){ cudaFreeArray(d_roiArr[L]); d_roiArr[L]=nullptr; }
    }
    if(d_img){ cudaFree(d_img); d_img=nullptr; g_imgBytes=0; }

    g_nx=nx; g_ny=ny; g_nz=nz;
    if(!uploadVolume(t1,nx,ny,nz,d_t1Arr,h_t1Tex)) return -1;
    for(int L=0;L<n_labels;L++){
        const float* fld = roi_fields + (size_t)L*nx*ny*nz;
        if(!uploadVolume(fld,nx,ny,nz,d_roiArr[L],h_roiTex[L])) return -1;
    }
    return 0;
}

void gpu_free()
{
    if(h_t1Tex){ cudaDestroyTextureObject(h_t1Tex); h_t1Tex=0; }
    if(d_t1Arr){ cudaFreeArray(d_t1Arr); d_t1Arr=nullptr; }
    for(int L=0;L<MAXL;L++){
        if(h_roiTex[L]){ cudaDestroyTextureObject(h_roiTex[L]); h_roiTex[L]=0; }
        if(d_roiArr[L]){ cudaFreeArray(d_roiArr[L]); d_roiArr[L]=nullptr; }
    }
    if(d_img){ cudaFree(d_img); d_img=nullptr; g_imgBytes=0; }
}

// Same signature as the CPU version (t1/roi pointers are ignored: data already
// lives on the GPU after gpu_init), so the Python wrapper barely changes.
int render(const float* t1, int nx, int ny, int nz,
           double sx, double sy, double sz,
           const float* roi_fields, int n_labels, const float* label_colors,
           double azim, double elev,
           int show_vr, int show_sr, int show_roi, int show_clip,
           double t1_lo, double t1_hi,
           double sr_iso, double vr_thr, double vr_opacity,
           double clip_k,
           double zoom, double pan_u, double pan_v,
           unsigned char* out, int W, int H)
{
    (void)t1; (void)roi_fields;
    if(!h_t1Tex) return -1;                  // gpu_init not called
    if(nx!=g_nx||ny!=g_ny||nz!=g_nz) return -3;

    size_t need = (size_t)W*H*3;
    if(need > g_imgBytes){
        if(d_img) cudaFree(d_img);
        if(cudaMalloc(&d_img, need)!=cudaSuccess) return -4;
        g_imgBytes = need;
    }

    Params P;
    P.nx=nx; P.ny=ny; P.nz=nz;
    P.sx=(float)sx; P.sy=(float)sy; P.sz=(float)sz;
    P.n_labels = n_labels>MAXL? MAXL : n_labels;
    for(int L=0;L<P.n_labels;L++){
        P.roiTex[L]=h_roiTex[L];
        P.lc[3*L+0]=label_colors[3*L+0];
        P.lc[3*L+1]=label_colors[3*L+1];
        P.lc[3*L+2]=label_colors[3*L+2];
    }
    for(int L=P.n_labels;L<MAXL;L++){ P.roiTex[L]=0; P.lc[3*L]=P.lc[3*L+1]=P.lc[3*L+2]=0.f; }
    P.t1Tex=h_t1Tex;
    P.azim=(float)azim; P.elev=(float)elev;
    P.show_vr=show_vr; P.show_sr=show_sr; P.show_roi=show_roi; P.show_clip=show_clip;
    P.t1_lo=(float)t1_lo; P.t1_hi=(float)t1_hi;
    P.sr_iso=(float)sr_iso; P.vr_thr=(float)vr_thr; P.vr_opacity=(float)vr_opacity;
    P.clip_k=(float)clip_k;
    P.zoom=(float)zoom; P.pan_u=(float)pan_u; P.pan_v=(float)pan_v;
    P.W=W; P.H=H;

    dim3 blk(16,16);
    dim3 grd((W+blk.x-1)/blk.x, (H+blk.y-1)/blk.y);
    kRender<<<grd,blk>>>(P, d_img);
    if(cudaGetLastError()!=cudaSuccess) return -5;
    if(cudaDeviceSynchronize()!=cudaSuccess) return -6;
    if(cudaMemcpy(out, d_img, need, cudaMemcpyDeviceToHost)!=cudaSuccess) return -7;
    return 0;
}

} // extern "C"


---
## 3. GPU の確認とコンパイル

C++のソースを `nvcc` でコンパイルし、Pythonから呼べる共有ライブラリ (`.so`) を作ります。

**なぜGPUなのか？** レイキャスティング法の弱点は講義で述べた通り「計算量が多い」ことです。
しかし、各画素の光線計算は**互いに独立**しています。
GPUは数千のコアで**1画素=1スレッド**として同時に処理できるため、この種の計算と極めて相性が良いのです。
（640×640画像なら、約41万本の光線を並列に飛ばしています。）

> エラーが出る場合は、GPUランタイムが選択されていません。上部メニューから設定してください。

In [ ]:
import subprocess, shutil

if shutil.which('nvidia-smi') is None:
    raise SystemExit(
        'GPU が割り当てられていません。\n'
        '  ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = T4 GPU → 保存\n'
        '  ※ 切り替えるとランタイムが再起動するので、セル1から実行し直してください。')

!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

cc = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().strip().split('\n')[0].replace('.', '')
ARCH = f'sm_{cc}'
print('building for', ARCH)

!nvcc -O3 -Xcompiler -fPIC -shared -arch={ARCH} renderer.cu -o librenderer.so && echo 'Build OK'

---
## 4. データの読み込み

### ここで行っていること

1. **T1c画像の読み込み**（ダウンサンプリングせず、**オリジナル解像度のまま**扱います）
2. **ボクセルサイズ (spacing) をヘッダーから取得** → `spacing=(1.0, 1.0, 1.2) mm` のように表示されます。
   この値がC++側に渡り、正しいアスペクト比（物理座標系）を保ちます。
3. **信号強度のウィンドウ設定**（1〜99パーセンタイル）。CTのウィンドウ幅/レベルと同じ発想です。
4. **ROIの前処理**：ラベル値ごとにマスクを分離し、**ガウシアンフィルタで平滑化**します。

> **なぜROIを平滑化するのか？**
> マスクは 0/1 の階段状データなので、そのまま等値面を描くとボクセルの角が
> ゴツゴツと見えてしまいます（**エイリアシング**）。平滑化して連続的な値にしてから
> 0.5 の等値面を描くことで、なめらかな臓器表面が得られます（`SMOOTH_SIGMA` で調整可）。

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# 例: T1_PATH = '/content/drive/MyDrive/.../T1c.nii.gz'

T1_PATH  = 'T1c.nii.gz'
ROI_PATH = 'tumor_ROI.nii.gz'

import numpy as np, ctypes, nibabel as nib, matplotlib
from scipy.ndimage import gaussian_filter

# --- T1 ---
t1_img = nib.load(T1_PATH)
t1 = np.ascontiguousarray(t1_img.get_fdata(), dtype=np.float32)
nx, ny, nz = t1.shape
sx, sy, sz = [float(z) for z in t1_img.header.get_zooms()[:3]]
print(f'T1  shape={t1.shape}  spacing=({sx:.3f}, {sy:.3f}, {sz:.3f}) mm')

pos = t1[t1 > 0]
if pos.size: T1_LO, T1_HI = map(float, np.percentile(pos, [1, 99]))
else:        T1_LO, T1_HI = float(t1.min()), float(t1.max())
print(f'intensity window: [{T1_LO:.1f}, {T1_HI:.1f}]')

# --- ROI (multi-label) -> per-label smoothed field ---
roi_img = nib.load(ROI_PATH)
roi = np.ascontiguousarray(np.rint(roi_img.get_fdata()), dtype=np.float32)
assert roi.shape == t1.shape, f'ROI {roi.shape} != T1 {t1.shape}'
labels = [int(l) for l in np.unique(roi) if l != 0]
print('ROI labels:', labels)

SMOOTH_SIGMA = 1.2
fields = [gaussian_filter((roi == L).astype(np.float32), SMOOTH_SIGMA) for L in labels]
roi_fields = np.ascontiguousarray(
    np.stack(fields, 0) if fields else np.zeros((0, nx, ny, nz), np.float32), dtype=np.float32)
n_labels = len(labels)

pal = matplotlib.colormaps['tab10']
label_colors = np.ascontiguousarray(
    np.array([pal(i % 10)[:3] for i in range(max(n_labels, 1))], dtype=np.float32))
print('label colors:\n', np.round(label_colors, 2))

---
## 5. PythonとC++の接続（ctypes）

`ctypes` を使って、コンパイルした `.so` の関数をPythonから直接呼び出します。

- **`gpu_init()`** … ボリュームデータを **GPUのテクスチャメモリに1回だけ転送**します。
  マウスを動かすたびに数百MBを転送していては話にならないため、データはGPU上に常駐させます。
  テクスチャメモリを使うと、**トリリニア補間がハードウェアで自動的に行われる**という利点もあります
  （光線が通る位置は必ずしもボクセルの中心ではないため、補間が必須です）。
- **`render()`** … 視点・閾値・スライス位置などを渡して1枚描画します。

In [ ]:
import ctypes
import numpy as np

for _v in ('t1','roi_fields','label_colors','nx','ny','nz','sx','sy','sz',
           'n_labels','T1_LO','T1_HI'):
    if _v not in globals():
        raise SystemExit(f"'{_v}' が未定義です。先にセル4（データ読み込み）を実行してください。")

lib = ctypes.CDLL('./librenderer.so')
D, I, FP, UP = ctypes.c_double, ctypes.c_int, ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_ubyte)

lib.gpu_init.argtypes = [FP, I, I, I, FP, I, FP]
lib.gpu_init.restype  = I
lib.render.argtypes = [FP, I, I, I, D, D, D, FP, I, FP, D, D,
                       I, I, I, I, D, D, D, D, D, D,
                       D, D, D,            # zoom, pan_u, pan_v
                       UP, I, I]
lib.render.restype  = I
lib.gpu_free.restype = None

def _fp(a): return a.ctypes.data_as(FP)

rc = lib.gpu_init(_fp(t1), nx, ny, nz, _fp(roi_fields), n_labels, _fp(label_colors))
if rc != 0:
    raise RuntimeError(f'gpu_init failed (code {rc}) — GPUメモリ不足 or ラベル数>8')
print(f'GPU upload OK: {(t1.nbytes + roi_fields.nbytes)/1024**2:.1f} MB '
      f'({n_labels} labels) -> 3D textures')

def render(azim, elev, show_vr, show_sr, show_roi, show_clip, clip_k,
           sr_iso=0.35, vr_thr=0.12, vr_opacity=0.6,
           zoom=1.0, pan_u=0.0, pan_v=0.0, W=512, H=512):
    out = np.zeros((H, W, 3), dtype=np.uint8)
    r = lib.render(_fp(t1), nx, ny, nz, sx, sy, sz,
                   _fp(roi_fields), n_labels, _fp(label_colors),
                   float(azim), float(elev),
                   int(show_vr), int(show_sr), int(show_roi), int(show_clip),
                   T1_LO, T1_HI, float(sr_iso), float(vr_thr), float(vr_opacity),
                   float(clip_k), float(zoom), float(pan_u), float(pan_v),
                   out.ctypes.data_as(UP), W, H)
    if r != 0:
        raise RuntimeError(f'render failed (code {r})')
    return out

---
## 6. 🖱️ インタラクティブ・ビューワ（演習の本体）

下のセルを実行すると、マウスで操作できる3Dビューワが起動します。

### 操作方法

| 操作 | 動作 |
|---|---|
| **左ドラッグ** | 回転 |
| **ホイール** | 拡大 / 縮小 |
| **右ドラッグ（上下）** | **スライス移動**（切断面が前後に動く） |
| **Shift + ドラッグ** | 平行移動 |
| **ダブルクリック** | 視点リセット |

### 表示要素（チェックボックス）

| 項目 | 内容 |
|---|---|
| **T1 Volume Rendering** | 半透明のVR。内部構造が透けて見える |
| **T1 Surface Rendering** | 閾値による不透明な等値面。脳の「表面」だけが見える |
| **Tumor ROI Surface** | 腫瘍マスクをラベルごとに色分けしたサーフェス |
| **Clip plane (2D slice)** | **仮想切除**：ボリュームをスライス位置で切断し、切断面に2D断層像を貼る |

### パラメータ（スライダー）

- **SR iso** … サーフェスレンダリングの閾値。**これが表面の形を決めます**。
- **VR thr / VR opacity** … 伝達関数の形。どの信号強度から、どれくらい不透明にするか。

> **表示のコツ**：腫瘍を見たいときは **T1 Surface Rendering を OFF** にしてください。
> 不透明な脳表面に隠れてしまいます。**VR を ON にすると、半透明の脳の中に腫瘍が透けて見えます** —
> これがまさに、VRとSRの決定的な違いです。

> 💡 ドラッグ中は低解像度で高速に追従し、**マウスを離した瞬間に高解像度**で描き直します。
> 画像の下に、GPUの描画時間 (ms) が表示されます。

In [ ]:
import io, json, base64, time
import numpy as np
from PIL import Image
from IPython.display import display, HTML, JSON
from google.colab import output

FULL_RES  = 640    # ボタンを離した後の高解像度
DRAFT_RES = 288    # ドラッグ中の低解像度（軽快な追従用）

def _render_rpc(state_json, draft):
    """JS から呼ばれる: 状態を受け取り CUDA で描画して画像を返す"""
    try:
        s = json.loads(state_json)
        draft = bool(int(draft))
        res = DRAFT_RES if draft else FULL_RES
        t0 = time.time()
        img = render(
            azim=s['azim'], elev=s['elev'],
            show_vr=s['vr'], show_sr=s['sr'], show_roi=s['roi'], show_clip=s['clip'],
            clip_k=s['slice'],
            sr_iso=s['iso'], vr_thr=s['thr'], vr_opacity=s['op'],
            zoom=s['zoom'], pan_u=s['panU'], pan_v=s['panV'],
            W=res, H=res)
        ms = (time.time() - t0) * 1000
        buf = io.BytesIO()
        if draft:                       # ドラッグ中は軽い JPEG
            Image.fromarray(img).save(buf, format='JPEG', quality=85)
            mime = 'jpeg'
        else:                           # 確定時は劣化のない PNG
            Image.fromarray(img).save(buf, format='PNG', compress_level=1)
            mime = 'png'
        uri = f"data:image/{mime};base64," + base64.b64encode(buf.getvalue()).decode()
        # Colab のコールバックは MIME バンドルを返す必要がある
        return JSON({'img': uri, 'ms': round(ms), 'res': res})
    except Exception as e:
        import traceback; traceback.print_exc()
        return JSON({'error': repr(e)})

output.register_callback('rc.render', _render_rpc)

_HTML = """
<style>
  #rcWrap{font-family:system-ui,sans-serif;color:#ddd;background:#111;padding:10px;border-radius:8px;display:inline-block}
  #rcCanvas{background:#000;border:1px solid #333;border-radius:4px;cursor:grab;display:block}
  #rcCanvas.drag{cursor:grabbing}
  #rcBar{margin:8px 0;display:flex;flex-wrap:wrap;gap:14px;align-items:center;font-size:13px}
  #rcBar label{cursor:pointer;user-select:none}
  #rcSliders{display:flex;flex-wrap:wrap;gap:14px;font-size:12px;margin-bottom:8px}
  #rcSliders input[type=range]{width:110px;vertical-align:middle}
  #rcStat{font-size:12px;color:#8bc34a;font-family:monospace;min-height:16px}
  #rcHelp{font-size:11px;color:#888;margin-top:6px;line-height:1.5}
  button#rcReset{background:#333;color:#ddd;border:1px solid #555;border-radius:4px;padding:3px 10px;cursor:pointer}
</style>
<div id="rcWrap">
  <div id="rcBar">
    <label><input type="checkbox" id="cbVR"  checked> T1 Volume Rendering</label>
    <label><input type="checkbox" id="cbSR"> T1 Surface Rendering</label>
    <label><input type="checkbox" id="cbROI" checked> Tumor ROI Surface</label>
    <label><input type="checkbox" id="cbCLIP" checked> Clip plane (2D slice)</label>
    <button id="rcReset">Reset view</button>
  </div>
  <div id="rcSliders">
    <span>Slice Z <input type="range" id="sSlice" min="0" max="__NZ__" value="__Z0__"> <b id="vSlice">__Z0__</b></span>
    <span>SR iso <input type="range" id="sIso" min="5" max="90" value="35"> <b id="vIso">0.35</b></span>
    <span>VR thr <input type="range" id="sThr" min="0" max="80" value="12"> <b id="vThr">0.12</b></span>
    <span>VR opacity <input type="range" id="sOp" min="10" max="100" value="60"> <b id="vOp">0.60</b></span>
  </div>
  <canvas id="rcCanvas" width="640" height="640"></canvas>
  <div id="rcStat">initialising…</div>
  <div id="rcHelp">
    🖱️ <b>左ドラッグ</b>=回転 &nbsp;|&nbsp; <b>ホイール</b>=ズーム &nbsp;|&nbsp;
    <b>右ドラッグ(上下)</b>=スライス移動 &nbsp;|&nbsp; <b>Shift+ドラッグ / 中ボタン</b>=平行移動 &nbsp;|&nbsp;
    <b>ダブルクリック</b>=リセット
  </div>
</div>
<script>
(function(){
  const NZ = __NZ__;
  const cv  = document.getElementById('rcCanvas');
  const ctx = cv.getContext('2d');
  const stat= document.getElementById('rcStat');

  const DEF = {azim:2.4, elev:0.35, zoom:1.0, panU:0.0, panV:0.0, slice:Math.round(NZ/2)};
  let S = Object.assign({}, DEF, {vr:1, sr:0, roi:1, clip:1, iso:0.35, thr:0.12, op:0.60});

  // Colab の返り値から payload を安全に取り出す（MIME バンドルの揺れに耐える）
  function payload(r){
    if(!r) return null;
    const d = r.data || r;
    if(!d) return null;
    if(d['application/json']) return d['application/json'];
    if(d['text/plain']){
      try{ return JSON.parse(d['text/plain'].replace(/^['"]|['"]$/g, '')); }
      catch(e){ return {error: 'unexpected payload: ' + d['text/plain'].slice(0,200)}; }
    }
    return {error: 'no json payload: ' + JSON.stringify(Object.keys(d)).slice(0,200)};
  }

  // ---- RPC (throttled: one render in flight at a time) ----
  let inFlight = false, queued = false, queuedDraft = true;
  const bitmap = new Image();
  bitmap.onload = () => {
    ctx.clearRect(0,0,cv.width,cv.height);
    ctx.imageSmoothingEnabled = true;
    ctx.drawImage(bitmap, 0, 0, cv.width, cv.height);   // 低解像度でも同サイズに拡大
  };

  async function draw(draft){
    if(inFlight){ queued = true; queuedDraft = queuedDraft && draft; return; }
    inFlight = true;
    try{
      const r = await google.colab.kernel.invokeFunction(
                  'rc.render', [JSON.stringify(S), draft ? 1 : 0], {});
      const d = payload(r);
      if(!d || d.error){
        stat.textContent = 'ERROR: ' + (d ? d.error : 'empty response');
        stat.style.color = '#e57373';
      }else{
        bitmap.src = d.img;
        stat.style.color = draft ? '#ffb74d' : '#8bc34a';
        stat.textContent =
          `Z=${S.slice}  zoom=${S.zoom.toFixed(2)}  `
          + `azim=${S.azim.toFixed(2)} elev=${S.elev.toFixed(2)}  |  `
          + `GPU ${d.ms} ms @ ${d.res}px ${draft ? '(draft)' : '(full)'}`;
      }
    }catch(e){ stat.textContent = 'RPC error: ' + e; stat.style.color='#e57373'; }
    inFlight = false;
    if(queued){ const q = queuedDraft; queued = false; queuedDraft = true; draw(q); }
  }

  // ---- mouse ----
  let mode = null, lx = 0, ly = 0;
  const MODE = {ROT:'rot', SLICE:'slice', PAN:'pan'};

  cv.addEventListener('contextmenu', e => e.preventDefault());

  cv.addEventListener('mousedown', e => {
    e.preventDefault();
    if(e.button === 2)                     mode = MODE.SLICE;   // 右 = スライス
    else if(e.button === 1 || e.shiftKey)  mode = MODE.PAN;     // 中 or Shift = パン
    else                                   mode = MODE.ROT;     // 左 = 回転
    lx = e.clientX; ly = e.clientY;
    cv.classList.add('drag');
  });

  window.addEventListener('mousemove', e => {
    if(!mode) return;
    const dx = e.clientX - lx, dy = e.clientY - ly;
    lx = e.clientX; ly = e.clientY;
    if(mode === MODE.ROT){
      S.azim -= dx * 0.010;
      S.elev  = Math.max(-1.5, Math.min(1.5, S.elev + dy * 0.008));
    }else if(mode === MODE.SLICE){
      S.slice = Math.max(0, Math.min(NZ, S.slice - dy * 0.5));
      const z = Math.round(S.slice);
      document.getElementById('sSlice').value = z;
      document.getElementById('vSlice').textContent = z;
    }else if(mode === MODE.PAN){
      S.panU -= dx / cv.width  * 1.1 / S.zoom;
      S.panV += dy / cv.height * 1.1 / S.zoom;
    }
    draw(true);                            // ドラッグ中は draft
  });

  window.addEventListener('mouseup', () => {
    if(!mode) return;
    mode = null;
    cv.classList.remove('drag');
    draw(false);                           // 離したら full-res で確定
  });

  cv.addEventListener('wheel', e => {      // ホイール = ズーム
    e.preventDefault();
    S.zoom = Math.max(0.3, Math.min(12.0, S.zoom * Math.exp(-e.deltaY * 0.0015)));
    draw(true);
    clearTimeout(cv._wt);
    cv._wt = setTimeout(() => draw(false), 260);   // 停止したら高解像度
  }, {passive:false});

  cv.addEventListener('dblclick', e => {   // ダブルクリック = リセット
    e.preventDefault();
    Object.assign(S, DEF);
    document.getElementById('sSlice').value = DEF.slice;
    document.getElementById('vSlice').textContent = DEF.slice;
    draw(false);
  });

  // ---- checkboxes / sliders ----
  const bind = (id, key) => document.getElementById(id)
      .addEventListener('change', e => { S[key] = e.target.checked ? 1 : 0; draw(false); });
  bind('cbVR','vr'); bind('cbSR','sr'); bind('cbROI','roi'); bind('cbCLIP','clip');

  const slide = (id, vid, key, scale, dec) => {
    const el = document.getElementById(id), lab = document.getElementById(vid);
    el.addEventListener('input', e => {
      S[key] = e.target.value * scale;
      lab.textContent = (dec === 0) ? Math.round(S[key]) : S[key].toFixed(dec);
      draw(true);
    });
    el.addEventListener('change', () => draw(false));
  };
  slide('sSlice','vSlice','slice', 1,    0);
  slide('sIso',  'vIso',  'iso',   0.01, 2);
  slide('sThr',  'vThr',  'thr',   0.01, 2);
  slide('sOp',   'vOp',   'op',    0.01, 2);

  document.getElementById('rcReset').addEventListener('click', () => {
    Object.assign(S, DEF);
    document.getElementById('sSlice').value = DEF.slice;
    document.getElementById('vSlice').textContent = DEF.slice;
    draw(false);
  });

  draw(false);   // 初回描画
})();
</script>
"""

display(HTML(_HTML.replace('__NZ__', str(nz - 1)).replace('__Z0__', str(nz // 2))))

---
## 7. 演習課題

ビューワを操作しながら、以下を確認してください。

### 課題1：ボリュームレンダリング vs サーフェスレンダリング

1. **VRだけをON**にして観察する。
2. 次に **SRだけをON**にして観察する。
3. **問い**：腫瘍 (ROI) を表示したまま、SRだけをONにすると腫瘍はどうなりますか？
   VRではどうですか？ **なぜそうなるのか**、講義の「長所・短所」の表と対応づけて説明しなさい。

### 課題2：閾値の恐ろしさ（サーフェスレンダリング）

1. **SRのみON**にする。
2. **SR iso** スライダーを 0.05 → 0.90 までゆっくり動かす。
3. **問い**：閾値を下げすぎるとどうなりますか？ 上げすぎるとどうなりますか？
   「閾値の設定一つで結果が大きく変わってしまう」という短所を、具体的に記述しなさい。
4. **考察**：この性質は、3Dプリンタ用のSTLデータを作る場面ではどんなリスクになりますか？

### 課題3：伝達関数の設計（ボリュームレンダリング）

1. **VRのみON**にする。
2. **VR thr** と **VR opacity** を動かし、脳実質が最もよく見える設定を探す。
3. **問い**：「伝達関数の設定が職人技になりがち」と言われる理由を、体験に基づいて説明しなさい。
4. **発展**：本実装の伝達関数は `a = opacity × frac²` という単純な2次関数です。
   臨床用ソフトではどのような伝達関数が使われているか調べなさい。

### 課題4：手術シミュレーション（仮想切除）

1. **Clip plane を ON**、**VR も ON** にする。
2. **右ドラッグ（上下）** で切断面を腫瘍の位置まで動かす。
3. **問い**：切断面に表示されている「クッキリした画像」は何ですか？ 3Dのどの情報から来ていますか？
4. **考察**：この「切断して断面を見る」機能は、講義2.2で述べた**仮想肝切除**や
   **脳腫瘍切除の治療計画**において、なぜ有用なのか説明しなさい。

### 課題5：物理座標系の重要性

セル4の出力に表示された `spacing` の値を確認してください。

- **問い**：もしプログラムがボクセルサイズを無視し、すべてのボクセルを立方体 (1,1,1) として
  扱った場合、表示される脳の形はどうなりますか？
- **発展**：セル5の `render()` 呼び出しで、`sz` を `sz*2` に変えて描画し、実際に確かめなさい。
  手術ナビゲーションにおいて、この歪みが起きたら何が問題になりますか？（講義3.1と関連づけて）

### 発展課題（余力のある人へ）

1. **MIP（最大値投影法）を実装する**：`renderer.cu` のVRループを書き換え、色を合成する代わりに
   光線上の**最大値**を記録するようにすれば、MIPになります。血管が強調されることを確認しなさい。
   （ヒント：`accR = fmaxf(accR, nv);` として、`accA` の蓄積をやめる）
2. **透視投影法に変更する**：全画素で共通だった光線方向 `dir` を、
   視点から画素へ向かう方向に変えると透視投影になります。**仮想内視鏡**への第一歩です。
3. **矢状断・冠状断でのクリッピング**：現在の切断面はZ軸（Axial）に垂直です。
   平面の法線を変えて、Sagittal / Coronal で切れるようにしなさい。

---
## 8. まとめ

本演習では、講義で学んだ3D可視化アルゴリズムを実際に動かしました。

| 学んだこと | 実装で確認したこと |
|---|---|
| レイキャスティング法 | 各画素から光線を飛ばし、色と不透明度を合成する |
| ボリュームレンダリング | 透明度により**内部構造が見える**。伝達関数の設計が肝 |
| サーフェスレンダリング | 高速で形状把握に優れるが、**閾値依存**で**内部が見えない** |
| 手術シミュレーション | 3Dモデルの**仮想切除**と、断面の2D画像との対応づけ |
| 座標系 | ボクセルサイズを考慮しなければ、形状が歪む |

これらの技術が統合されることで、**術前計画 → 術中ナビゲーション → 低侵襲治療**という
一連の治療プロセスが支えられています。

「3Dできれいに見える」ことの裏側には、**どの閾値を選んだか、どんな伝達関数を設定したか**という
**人間の判断**が必ず入っています。臨床で3D画像を扱う際は、
**その画像がどう作られたかを疑える技術者**であってください。

---
## 付録A：予備UI（ipywidgets スライダー版）

上のマウス操作ビューワはColab専用のJavaScript連携を使っています。
Colab以外のJupyter環境で動かす場合や、canvasが表示されない場合は、
こちらの従来型スライダーUIを使ってください（機能は同等です）。

In [ ]:
import ipywidgets as wg
import matplotlib.pyplot as plt
from IPython.display import display

cb_vr   = wg.Checkbox(value=True,  description='T1 Volume Rendering')
cb_sr   = wg.Checkbox(value=False, description='T1 Surface Rendering')
cb_roi  = wg.Checkbox(value=True,  description='Tumor ROI Surface')
cb_clip = wg.Checkbox(value=True,  description='Clip plane (2D slice)')
s_clip = wg.IntSlider(value=nz//2, min=0, max=nz-1, description='Slice Z')
s_azim = wg.FloatSlider(value=2.4, min=0, max=6.28, step=0.02, description='Azimuth')
s_elev = wg.FloatSlider(value=0.35, min=-1.5, max=1.5, step=0.02, description='Elevation')
s_zoom = wg.FloatSlider(value=1.0, min=0.3, max=8.0, step=0.05, description='Zoom')
s_iso  = wg.FloatSlider(value=0.35, min=0.05, max=0.9, step=0.01, description='SR iso')
s_thr  = wg.FloatSlider(value=0.12, min=0.0, max=0.8, step=0.01, description='VR thr')
s_op   = wg.FloatSlider(value=0.60, min=0.1, max=1.0, step=0.05, description='VR opacity')
out_img = wg.Output()

def redraw(*_):
    with out_img:
        out_img.clear_output(wait=True)
        img = render(s_azim.value, s_elev.value, cb_vr.value, cb_sr.value,
                     cb_roi.value, cb_clip.value, s_clip.value,
                     s_iso.value, s_thr.value, s_op.value,
                     zoom=s_zoom.value, W=512, H=512)
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        ax.imshow(img); ax.axis('off'); plt.show()

for c in [cb_vr, cb_sr, cb_roi, cb_clip, s_clip, s_azim, s_elev, s_zoom, s_iso, s_thr, s_op]:
    c.observe(redraw, 'value')

display(wg.VBox([wg.HBox([cb_vr, cb_sr, cb_roi, cb_clip]),
                 wg.HBox([s_clip, s_azim, s_elev, s_zoom]),
                 wg.HBox([s_iso, s_thr, s_op]), out_img]))
redraw()

---
## 付録B：トラブルシューティング

| 症状 | 原因と対処 |
|---|---|
| `GPU が割り当てられていません` | ランタイム → ランタイムのタイプを変更 → **T4 GPU**。切替後は**セル1から再実行**。 |
| `FileNotFoundError: T1c.nii.gz` | ファイルがColabにアップロードされていません。左のファイルペインにドラッグするか、Driveをマウントしてパスを修正。 |
| `'t1' が未定義です` | セル4（データ読み込み）が未実行です。上から順に実行してください。 |
| `gpu_init failed (code -2)` | ROIのラベル数が8を超えています（`MAXL` の上限）。 |
| `render failed (code -1)` | `gpu_init` が実行されていません（セル5を再実行）。 |
| ビューワが真っ黒 | 全チェックボックスがOFF、または `SR iso` / `VR thr` が高すぎます。**Reset view** を押して、SR iso を下げてください。 |
| 動作が重い | セル6の `FULL_RES` (既定640) と `DRAFT_RES` (既定288) を下げてください。 |
| `ERROR:` が赤字で出る | 表示された文言と、セル下のPythonトレースバックを確認してください。 |